This is the first part of a talk at GOSIM 2026, given together with Olivier Grisel (who
does a demo in the second half of this talk).
## The Adoption of Python Array API in scikit-learn

### The goal of introducing Array API in scikit-learn

#### Slide 1

- scikit-learn is seasoned a machine learning library that was build with running on
  numpy arrays on CPUs in mind

- with more people using GPUs for ML and AI, more array libraries and backends come into
  play

- since version 1.3 (released in June 2023) we implement support in a growing number or
  functions and estimators in experimental mode

#### Slide 2

- in the Python ML ecosystem, scikit-learn is an array consuming library

    - a library that gets arrays as inputs and outputs arrays

    - the goal is that during the operations within scikit-learn, the arrays stay on the
      namespace and on the device (specifically GPU devices) they came from for all of
      the computationally intense tasks and for most of the tasks in general
      
    - as a consequence, users in addition to NumPy array can pass CuPy arrays and 
    PyTorch tensors; JAX support is in preparation


### The Python Array API Standard

#### Slide 3

- *Consortium for Python Data API Standards* (consisting of maintainers of array heavy
  Python libraries) developed a few tools that array libraries and array consuming
  libraries such like scikit-learn can use for development

    - [`Array API standard`](https://github.com/data-apis/array-api): defines data
      types, operations and behaviours and publishes a
      [SPEC](https://data-apis.org/array-api/latest/index.html) on it; the spec is versioned and still evolves

    - [`array-api-tests`](https://github.com/data-apis/array-api-tests) suite where
      array libraries can test if they comply to the standard

    - [`array-api-strict`](https://github.com/data-apis/array-api-strict) mimics a
      library that solely emulates the array API standard
        - testing library used to check if we support the minimum standard (even though
          torch and CuPy mostly support additional functionality) on its emulated
          hardware (which makes it runnable easily on the CI)
        - heavily used in scikit-learn CI

    - [`array_api_compat`](https://github.com/data-apis/array-api-compat) bridges gaps
      for parts of numpy, torch and cupy, that don't yet fully support the array API

    - [`array-api-extra`](https://github.com/data-apis/array-api-extra) provides array
      functions not included in the standard but found useful for array-consuming
      libraries

    - scikit-learn uses `array-api-strict` for testing and vendors both `array-api-compat`
      and `array-api-extra` since 1.7

### Passing array API compatible inputs into scikit-learn functions and estimators

#### Slide 4

- since scipy has array API support in experimental mode as well, users need to set 
  `export SCIPY_ARRAY_API=1`

- users currently need to enable array API support by setting a flag on the global
  configuration: </br> `from sklearn import set_config` </br>
  `set_config(array_api_dispatch=True)` </br> or by using the config context manager

- for a full list of supported estimators and functions, see [Array API support
  (experimental)](https://scikit-learn.org/stable/modules/array_api.html)

#### Slide 5

In [ ]:
# export SCIPY_ARRAY_API=1 # <-- set environment variable

from sklearn.datasets import make_classification
from sklearn import config_context
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import cupy

X_np, y_np = make_classification(random_state=0)
X_cu = cupy.asarray(X_np)
y_cu = cupy.asarray(y_np)
X_cu.device

with config_context(array_api_dispatch=True):
    lda = LinearDiscriminantAnalysis()
    X_trans = lda.fit_transform(X_cu, y_cu)
X_trans.device

### Implementation in scikit-learn and Challenges

#### Slide 6 
- main tracking issue: [Path for Adopting the Array API
  spec](https://github.com/scikit-learn/scikit-learn/issues/22352)

#### Slide 7

- scikit-learn is an array consuming library: inputs are arrays + outputs are arrays 

- principle: computations stay on the same device and namespace as the input arrays:

In [12]:
%%skip

from sklearn.utils._array_api import get_namespace, device

xp, is_array_api = get_namespace(X)  # same library as `X` (CuPy, torch, numpy, strict, …)
device = device(X)                   # same device as `X`

# inside estimators: 
# keep operations on namespace `xp` and 
# create new arrays on namespace `xp` and `device`

UsageError: Cell magic `%%skip` not found.


#### Slide 8

- the [`Array API standard`](https://github.com/data-apis/array-api) is a limited subset
  of NumPy functionality

  - limited subset of types and operations; for instance no support for sparse arrays
    or matrixes

  - not the whole width of functionality; for instance no u-funcs, sometimes need to
    re-write computational logic for arrays other than numpy

  - some higher order functions are re-created in scikit-learn, mostly by subcasing the
    behavior depending on the different backends and where the function is build from
    lower order function within the array API spec; for instance `_nanmin()`

In [4]:
def _nanmin(X, axis=None, xp=None):
    # TODO: refactor once nan-aware reductions are standardized:
    # https://github.com/data-apis/array-api/issues/621
    xp, _, device_ = get_namespace_and_device(X, xp=xp)
    if _is_numpy_namespace(xp):
        return xp.asarray(numpy.nanmin(X, axis=axis))

    else:
        mask = xp.isnan(X)
        X = xp.min(
            xp.where(mask, xp.asarray(+xp.inf, dtype=X.dtype, device=device_), X),
            axis=axis,
        )
        # Replace Infs from all NaN slices with NaN again
        mask = xp.all(mask, axis=axis)
        if xp.any(mask):
            X = xp.where(mask, xp.asarray(xp.nan, dtype=X.dtype, device=device_), X)
        return X

#### Slide 9

- commonly, scikit-learn estimators get input arrays that differ or change by step and
  stage

  - train and predict often on different machines

  - fit, predict, transform or a metric function may be called with new data from a
    different array library or device

  - string inputs (NumPy-only) need to be processed alongside GPU-arrays

#### Slide 10

We developed two principles: 
  
  1. convert estimator attributes to the namespace and device passed as `X` to
     `predict`, so `predict` sees the same namespace and device as the new batch

  - PR [Raise if fit and predict use different array API namespaces or devices
    (continued)](https://github.com/scikit-learn/scikit-learn/pull/33076)

  2. we explicitly support for mixed namespace and device inputs (up from version 1.9),
    which is a unique approach in array consuming libraries
      - `y`, `sample_weight`, etc, follow `X` 
        - `X` is the larger array where computation speed matters and the cost of moving
      it to another device would be larger; thus `y` moves to `X`
      - `y_true`, `sample_weight` and `labels` follow `y_pred` / `y_score`
        - `y_pred` is most likely an output of a scikit-learn function
        - issue [Automatically move y (and sample_weight) to the same device and
          namespace as X](https://github.com/scikit-learn/scikit-learn/issues/28668)


#### Slide 11

- mental model: `X` defines the context (namespace and device)
    - estimator picks namespace + device from `X`, and computations stay there

- in pipelines only `X` can be transformed step by step, `y` cannot
    - trick: `FunctionTransformer` step that moves feature data from CPU to GPU
    - if a pipeline step moves `X` to CUDA, `y` may still be on CPU on `fit` time
    - scikit-learn applies the "follows" rule here: `y` and `sample_weight` are converted when needed

- "everything follows X" is a UX rule, so users can think in workflows rather than synchronosing all the inputs

In [2]:
from functools import partial
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, TargetEncoder

pipeline = make_pipeline(
    TargetEncoder(),
    # Convert feature array `X` to Torch CUDA device
    FunctionTransformer(partial(torch.asarray, dtype="float32", device="cuda")),
    RidgeClassifier(solver="svd"),
)

NameError: name 'torch' is not defined

#### Slide 12

- computation on compiled code (Cython, C or C++) is optimised to run on CPUs and hard
  to bridge

- estimators that do the heavy part of the computation on compiled code can either 
  
  a) use the compiled path by copying data to CPU, then map results back to the input
  namespace and device

  b) use the compiled path only for NumPy inputs; otherwise follow an Array API Python path
     
- discovering if plugins can be an alternative


#### Slide 13

- maintainers often need to compare and benchmark multiple implementation options:

  - trade-offs between runtime speed and memory footprint

  - benchmarking on realistic workloads versus edge cases

  - define reliable rules
  
  - final choice often involves design discussions between several maintainers


#### Slide 14

- plans of moving out of experimental mode

- as a consuming library, scikit-learn depends on reliable upstream Array API
  implementations but we have an issue to track which internal API design requirements
  and guarantees in form of testing we need to define before we can make
  `array_api_dispatch=True` the default
    
  - issue [RFC a plan to move array API support out of
  experimental](https://github.com/scikit-learn/scikit-learn/issues/33444) to plan what
  "non-experimental" would mean for defaults, errors, and coordination with SciPy

### Sources

- Array API SPEC: [Python array API
  standard](https://data-apis.org/array-api/latest/index.html), Consortium for Python
  Data API Standards.

- scikit-learn's documentation on Array API implementation: [Array API support
  (experimental)](https://scikit-learn.org/dev/modules/array_api.html).

- issue [Path for Adopting the Array API
  spec](https://github.com/scikit-learn/scikit-learn/issues/22352)

- issue [RFC a plan to move array API support out of
  experimental](https://github.com/scikit-learn/scikit-learn/issues/33444)

### References

- Lucy Liu: [Update on array API adoption in
  scikit-learn](https://labs.quansight.org/blog/array-api-scikit-learn-2026), Quansight
  Labs Blog, March 6 2026.
  
- Thomas J. Fan: [Array API Support in
  scikit-learn](https://labs.quansight.org/blog/array-api-support-scikit-learn),
  Quansight Labs Blog, September 19 2023.